# Model Comparison: Decision Tree vs PSTH vs RBF SVM

Load saved results from the three classifiers and compare:
1. **PCA vs No PCA** (MCCV 10 runs)
2. **Generalized vs Subject-specific** (MCCV 10 runs)

**Note:** Run `decision_tree.ipynb`, `psth_classifier.ipynb`, and `rbf_svm.ipynb` first to generate the result files.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import os

# Load results from all three models
results_dir = "model_results"
model_files = {
    "Decision Tree": "decision_tree_results.json",
    "PSTH": "psth_classifier_results.json",
    "RBF SVM": "rbf_svm_results.json",
}

results = {}
for name, fname in model_files.items():
    path = os.path.join(results_dir, fname)
    if os.path.exists(path):
        with open(path) as f:
            results[name] = json.load(f)
        print(f"Loaded {name}")
    else:
        print(f"Warning: {path} not found. Run the corresponding notebook first.")

if not results:
    raise FileNotFoundError("No result files found. Run decision_tree, psth_classifier, and rbf_svm notebooks first.")

In [ ]:
# Extract means and stds for each model
def get_mean_std(arr):
    return np.mean(arr), np.std(arr)

model_names = list(results.keys())
n_models = len(model_names)
width = 0.25
colors = ["#3498db", "#2ecc71", "#e74c3c"]  # blue, green, red

## Figure 1: PCA vs No PCA

In [ ]:
# Figure 1: PCA vs No PCA - grouped bars (3 models per category)
fig, ax = plt.subplots(figsize=(8, 5))

x_cats = np.arange(2)  # With PCA, Without PCA
multipliers = np.arange(n_models) - (n_models - 1) / 2  # center bars: [-1, 0, 1] for 3 models

for i, name in enumerate(model_names):
    r = results[name]
    mean_with, std_with = get_mean_std(r["f1_with_pca"])
    mean_no, std_no = get_mean_std(r["f1_no_pca"])
    offset = width * multipliers[i]
    ax.bar(
        x_cats + offset,
        [mean_with, mean_no],
        width,
        yerr=[std_with, std_no],
        capsize=4,
        label=name,
        color=colors[i % len(colors)],
        edgecolor="black",
    )

ax.set_xticks(x_cats)
ax.set_xticklabels(["With PCA", "Without PCA"])
ax.set_ylabel("Weighted F1")
ax.set_title("PCA vs No PCA: Model Comparison (MCCV 10 runs)")
ax.legend(loc="upper right")
ax.set_ylim(0, 0.5)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Figure 2: Generalized vs Subject-specific

In [ ]:
# Figure 2: Generalized vs Subject-specific - grouped bars (3 models per category)
fig, ax = plt.subplots(figsize=(8, 5))

x_cats = np.arange(2)  # Generalized, Subject-specific
multipliers = np.arange(n_models) - (n_models - 1) / 2

for i, name in enumerate(model_names):
    r = results[name]
    mean_gen, std_gen = get_mean_std(r["f1_generalized"])
    mean_subj, std_subj = get_mean_std(r["f1_subject_specific"])
    offset = width * multipliers[i]
    ax.bar(
        x_cats + offset,
        [mean_gen, mean_subj],
        width,
        yerr=[std_gen, std_subj],
        capsize=4,
        label=name,
        color=colors[i % len(colors)],
        edgecolor="black",
    )

ax.set_xticks(x_cats)
ax.set_xticklabels(["Generalized", "Subject-specific"])
ax.set_ylabel("Weighted F1")
ax.set_title("Generalized vs Subject-specific: Model Comparison (MCCV 10 runs)")
ax.legend(loc="upper right")
ax.set_ylim(0, 0.8)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Summary table
print("--- PCA vs No PCA (mean ± std) ---")
for name in model_names:
    r = results[name]
    m1, s1 = get_mean_std(r["f1_with_pca"])
    m2, s2 = get_mean_std(r["f1_no_pca"])
    print(f"{name}: With PCA={m1:.4f}±{s1:.4f}, No PCA={m2:.4f}±{s2:.4f}")

print("\n--- Generalized vs Subject-specific (mean ± std) ---")
for name in model_names:
    r = results[name]
    m1, s1 = get_mean_std(r["f1_generalized"])
    m2, s2 = get_mean_std(r["f1_subject_specific"])
    print(f"{name}: Generalized={m1:.4f}±{s1:.4f}, Subject-specific={m2:.4f}±{s2:.4f}")